## Sequence CLassification Using BERT
BERT (Bidirectional Encoder Representations from Transformers) is one of the most influential models in modern Natural Language Processing (NLP). In this notebook, instead of training language models from scratch, we can reuse pretrained BERT models and adapt them for downstream tasks such as sequence classification. We will load a pretrained BERT model from Hugging Face and use it for sentiment classification by finetuning the base model.

### What Are We Going to Cover?

In this notebook, we will:

- Understand what BERT is and why it is powerful

- Install and set up the Hugging Face Transformers library

- Load a pretrained BERT model for text classification

- Tokenize text properly for BERT

- Fine-tune BERT on a sentiment classification dataset

- Run inference on new sentences using our fine-tuned model

- Interpret model outputs (logits, probabilities, labels)

- Apply BERT to a sentiment classification task

### Key Learnings

By the end of this notebook, you will be able to:

- Explain how BERT differs from traditional NLP models

- Load a pretrained BERT model using Hugging Face

- Use BERT for sentiment or intent classification

- Understand tokenization, attention masks, and logits

- Perform inference on custom text inputs

### Setup and Installation
Why Hugging Face for downloading the pre-trained model?

We will use Hugging Face, because it provides:

- Thousands of pretrained NLP models

- Easy-to-use APIs

- Production-ready tools

We will use:

`transformers` → model & tokenizer

`torch` → model execution

### Install & Import

We will use the Hugging Face transformers library, which provides the `BertForSequenceClassification` class, a pre-built model that already has the classification head attached.
**Note** - Make sure to change the run time to GPU

In [ ]:
!pip install transformers datasets torch

import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

When using BERT for classification, we do not look at all the output vectors. Instead, we focus on the very first token, known as the [CLS] token (Classification token).

- **The Pooler Layer**: BERT is designed so that the output vector of the [CLS] token acts as a "summary" of the entire sentence.

- **The Classifier Head**: We attach a standard Linear layer (a Feed-Forward Network) to this summary vector. This layer maps BERT's 768-dimensional output to the number of classes we have (e.g., 2 for Positive/Negative).

The token output is then fed into a Softmax classifier

**What Makes BERT Special?**

Traditional NLP models read text left to right or right to left.
BERT reads text in both directions at the same time.

Example sentence:

```I did not like the movie```

BERT understands that "not" affects "like", which is crucial for sentiment tasks.

**Pretraining vs Fine-Tuning**
Let us understand quickly the difference between a `pre-trained` model and a `fine-tuned` model:

**Pretraining**: BERT learns general language understanding from massive text

**Fine-tuning**: BERT is adapted to a specific task (e.g., sentiment classification)

In this notebook, we will not only use the pre-trained model, we will also fine-tuned on our dataset.

### Load the Pre-trained Model & Tokenizer
We use `bert-base-uncased` (the "uncased" means it treats "Apple" and "apple" as the same).

In [ ]:
model_name = "bert-base-uncased"

# 1. Load the Tokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)

# 2. Load the Model with a classification head (num_labels=2 for Binary)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Prepare the Dataset
For this notebook, we will use the IMDb Movie Reviews dataset that consists of movie reviews. We will perform sentiment analysis on this data.

In [ ]:
# Load a small subset of the IMDb dataset for speed
dataset = load_dataset("imdb")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Tokenize the data
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Create smaller subsets for training
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Understanding Tokenisation
**Why Tokenisation Matters**: BERT does not read raw text. It reads token IDs, which are numerical representations of words or subwords from its vocabulary.

For example:
```
"I love this movie" → [101, 1045, 2293, 2023, 3185, 102]
```

The tokenizer also produces:
- **input_ids**: The numerical token IDs that BERT understands
- **attention_mask**: A binary mask (1s and 0s) indicating which tokens are real vs. padding
- **token_type_ids**: Used for sentence-pair tasks to distinguish between sentences

Special tokens:
- `[CLS]` (token ID 101): Added at the start; its output is used for classification
- `[SEP]` (token ID 102): Added at the end to mark sentence boundaries
- `[PAD]`: Used to make all sequences the same length in a batch

### Tokenise a Sample Sentence

In [ ]:
text = "I really enjoyed this movie!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

inputs


{'input_ids': tensor([[ 101, 1045, 2428, 5632, 2023, 3185,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

Next, let us fine-tune our base BERT model on our movie review dataset

### Define Training Arguments
This is where we set the "rules" for the fine-tuning process. Key hyperparameters include:

- **learning_rate**: We use a small value (1e-4) to avoid "catastrophic forgetting" — overwriting the valuable language knowledge BERT learned during pretraining
- **num_train_epochs**: Number of complete passes through the training data
- **per_device_train_batch_size**: How many samples to process at once (limited by GPU memory)
- **weight_decay**: L2 regularization to prevent overfitting

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-4,      # Tiny learning rate to avoid destroying pre-trained weights
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01
)

### Train and Evaluate
The `Trainer` API handles the heavy lifting of the training loop, gradient descent, and evaluation. During fine-tuning:

1. The model's pretrained weights are updated (not frozen) to adapt to our task
2. The newly added classification head learns to map BERT's representations to sentiment labels
3. Gradients flow through the entire network, allowing end-to-end optimization

This is different from "feature extraction" where we would freeze BERT and only train the classifier head.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Start Fine-Tuning
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss


TrainOutput(global_step=375, training_loss=0.45236861165364584, metrics={'train_runtime': 305.8469, 'train_samples_per_second': 9.809, 'train_steps_per_second': 1.226, 'total_flos': 789333166080000.0, 'train_loss': 0.45236861165364584, 'epoch': 3.0})

### Inference Pipeline
This function performs three critical steps:

1. **Preprocessing**: Tokenizes the raw text into a format BERT understands.

2. **Forward Pass**: Feeds the tokens into the model without calculating gradients (torch.no_grad()).

3. **Post-processing**: Converts the model's raw output (logits) into a human-readable label.

In [ ]:
def predict_sentiment(text, model, tokenizer):
  # 1. Switch model to 'eval' mode (disables dropout layers)
  model.eval()

  # 2. Tokenize the input text
  # We use the same parameters as training: padding and truncation
  inputs = tokenizer(
      text,
      return_tensors="pt",
      padding=True,
      truncation=True,
      max_length=512
  )

  # Move inputs to the same device as the model (GPU or CPU)
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  # 3. Disable gradient calculation to save memory and speed up inference
  with torch.no_grad():
      outputs = model(**inputs)

  # 4. Get the raw 'logits' (scores) from the model
  logits = outputs.logits

  # 5. Apply Argmax to find the index of the highest score
  # 0 = Negative, 1 = Positive (typical for IMDb dataset)
  prediction = torch.argmax(logits, dim=-1).item()

  # Map the index back to a string label
  labels = {0: "NEGATIVE", 1: "POSITIVE"}
  return labels[prediction]

#### Inference Pipeline Usage

In [ ]:
custom_text = "I absolutely loved this movie! The acting was phenomenal."
result = predict_sentiment(custom_text, model, tokenizer)
print(f"Text: {custom_text}\nPrediction: {result}")

Text: I absolutely loved this movie! The acting was phenomenal.
Prediction: POSITIVE


#### Understanding the Inference Process
Now, let us discuss some technical nuances to understand the overall fine-tuning and inferencing process

1. `model.eval()` vs. `torch.no_grad()`:
It is a common mistake to think these do the same thing.
 - - `model.eval()`: This notifies specific layers (like Dropout and Batch Normalization) to behave differently. For example, during inference, you want to use the entire network, so Dropout must be turned off.
 - - `torch.no_grad()`: This tells PyTorch not to keep track of the "computational graph." Since we aren't doing backpropagation (learning), this drastically reduces memory usage and makes the prediction faster.

2. Logits to Probabilities
The model returns Logits, which are raw numbers (e.g., [-1.2, 2.5]). If you just need the answer, use argmax. If you need a confidence score (e.g., "98% Positive"), you must pass the logits through a Softmax function:
```
probabilities = torch.nn.functional.softmax(logits, dim=-1)
confidence = torch.max(probabilities).item()
print(f"Confidence: {confidence:.2%}")
```

3. Handling Device Consistency
If you trained your model on a GPU (cuda), your input text must also be converted to a "GPU tensor" before being fed into the model. The line ```inputs = {k: v.to(model.device) for k, v in inputs.items()}``` ensures that no matter where your model lives, the data goes there too.